In [35]:
from sklearn.linear_model import LinearRegression,LogisticRegression
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
df=pd.read_csv('iris_extended.csv')

df.head()

,species,elevation,soil_type,sepal_length,sepal_width,petal_length,petal_width,sepal_area,petal_area,sepal_aspect_ratio,...,sepal_to_petal_length_ratio,sepal_to_petal_width_ratio,sepal_petal_length_diff,sepal_petal_width_diff,petal_curvature_mm,petal_texture_trichomes_per_mm2,leaf_area_cm2,sepal_area_sqrt,petal_area_sqrt,area_ratios
0,setosa,161.8,sandy,5.16,3.41,1.64,0.26,17.5956,0.4264,1.513196,...,3.146341,13.115385,3.52,3.15,5.33,18.33,53.21,4.194711,0.652993,41.265478
1,setosa,291.4,clay,5.48,4.05,1.53,0.37,22.1940,0.5661,1.353086,...,3.581699,10.945946,3.95,3.68,5.90,20.45,52.53,4.711051,0.752396,39.205087
2,setosa,144.3,sandy,5.10,2.80,1.47,0.38,14.2800,0.5586,1.821429,...,3.469388,7.368421,3.63,2.42,5.66,24.62,50.25,3.778889,0.747395,25.563910
3,setosa,114.6,clay,4.64,3.44,1.53,0.17,15.9616,0.2601,1.348837,...,3.032680,20.235294,3.11,3.27,4.51,22.91,50.85,3.995197,0.510000,61.367166
4,setosa,110.9,loamy,4.85,2.87,1.23,0.26,13.9195,0.3198,1.689895,...,3.943089,11.038462,3.62,2.61,4.03,21.56,40.57,3.730885,0.565509,43.525641


In [29]:
encoder=LabelEncoder()
df['species']=encoder.fit_transform(df['species'])
df['soil_type']=encoder.fit_transform(df['soil_type'])

In [30]:
features=df.drop('species',axis=1)
target=df['species']
X_train, X_test, y_train, y_test = train_test_split(features,target, test_size=0.35, random_state=42)

scaler = MinMaxScaler()
x_train=scaler.fit_transform(X_train)
x_test=scaler.transform(X_test)

In [36]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def ann_numpy_classifier(X_train, y_train, X_test, y_test, epochs=1000, hidden_size=8, learning_rate=0.01):
    np.random.seed(42)

    input_size = X_train.shape[1]
    output_size = y_train.shape[1]

    W1 = np.random.randn(input_size, hidden_size)
    b1 = np.zeros((1, hidden_size))

    W2 = np.random.randn(hidden_size, output_size)
    b2 = np.zeros((1, output_size))

    for epoch in range(epochs):
        Z1 = np.dot(X_train, W1) + b1
        A1 = sigmoid(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        loss = -np.mean(np.sum(y_train * np.log(A2 + 1e-8), axis=1))

        dZ2 = A2 - y_train
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * sigmoid_derivative(Z1)
        dW1 = np.dot(X_train.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # ---- Prediction ----
    def predict(X):
        A1 = sigmoid(np.dot(X, W1) + b1)
        A2 = softmax(np.dot(A1, W2) + b2)
        return np.argmax(A2, axis=1)

    y_pred = predict(X_test)
    y_true = np.argmax(y_test, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    accuracy = np.mean(y_pred == y_true)
    print(f"\nTest Accuracy: {accuracy:.4f}")

    return y_pred, (W1, b1, W2, b2)


In [32]:
from sklearn.preprocessing import OneHotEncoder

y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.values.reshape(-1, 1)

encoder_ohe = OneHotEncoder(sparse_output=False)
y_train_encoded = encoder_ohe.fit_transform(y_train_reshaped)
y_test_encoded = encoder_ohe.transform(y_test_reshaped)

ann_numpy_classifier(x_train, y_train_encoded, x_test, y_test_encoded, epochs=200, hidden_size=8, learning_rate=0.01)

Epoch 0, Loss: 1.4501
Epoch 10, Loss: 1.3698
Epoch 20, Loss: 0.6474
Epoch 30, Loss: 0.8167
Epoch 40, Loss: 0.8076
Epoch 50, Loss: 0.8006
Epoch 60, Loss: 0.8603
Epoch 70, Loss: 0.6288
Epoch 80, Loss: 0.7944
Epoch 90, Loss: 0.4556
Epoch 100, Loss: 0.0671
Epoch 110, Loss: 0.0492
Epoch 120, Loss: 0.0397
Epoch 130, Loss: 0.0338
Epoch 140, Loss: 0.0297
Epoch 150, Loss: 0.0266
Epoch 160, Loss: 0.0241
Epoch 170, Loss: 0.0222
Epoch 180, Loss: 0.0207
Epoch 190, Loss: 0.0194
Epoch 199, Loss: 0.0185

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       144
           1       0.99      0.99      0.99       143
           2       0.99      0.99      0.99       133

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Confusion Matrix:
[[144   0   0]
 [  0 142   1]
 [  0   1 132]]

Test Accuracy: 0.9952


(array([2, 2, 0, 1, 0, 2, 0, 2, 1, 2, 2, 0, 0, 2, 0, 0, 1, 1, 2, 1, 0, 1,
        1, 0, 0, 0, 1, 1, 0, 1, 0, 2, 1, 1, 0, 1, 1, 0, 0, 1, 2, 0, 2, 0,
        2, 0, 0, 0, 0, 0, 1, 2, 0, 0, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 2,
        0, 1, 1, 1, 1, 0, 2, 0, 0, 1, 0, 0, 2, 2, 2, 0, 2, 0, 1, 2, 2, 0,
        2, 1, 1, 1, 0, 0, 2, 1, 1, 2, 0, 0, 1, 0, 1, 0, 2, 1, 2, 2, 2, 2,
        0, 2, 0, 2, 1, 2, 0, 0, 1, 0, 1, 2, 2, 2, 1, 2, 0, 0, 1, 1, 0, 2,
        0, 0, 2, 1, 1, 1, 1, 2, 2, 1, 0, 1, 1, 1, 1, 1, 1, 1, 2, 0, 1, 0,
        2, 1, 0, 0, 2, 0, 2, 2, 2, 0, 0, 2, 0, 2, 2, 0, 1, 2, 0, 0, 0, 1,
        0, 2, 0, 1, 1, 2, 0, 2, 0, 2, 1, 2, 1, 1, 2, 1, 0, 2, 1, 0, 0, 2,
        2, 0, 2, 2, 1, 2, 2, 0, 1, 1, 1, 2, 2, 1, 1, 2, 2, 0, 1, 0, 1, 1,
        2, 2, 2, 2, 0, 1, 1, 0, 0, 1, 0, 0, 2, 1, 2, 0, 2, 1, 2, 2, 0, 1,
        0, 2, 0, 1, 1, 2, 0, 0, 1, 0, 0, 0, 2, 2, 0, 1, 0, 2, 1, 0, 2, 1,
        1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 0, 0, 0, 0, 1, 0, 2, 2, 0, 2, 1, 2,
        1, 2, 1, 2, 1, 2, 1, 2, 0, 1, 

In [34]:
model = Sequential([
    Dense(10, activation='relu', input_shape=(x_train.shape[1],)),
    Dense(y_train_encoded.shape[1], activation='softmax')  
])

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train_encoded, epochs=20, batch_size=8, verbose=1)

loss, accuracy = model.evaluate(x_test, y_test_encoded, verbose=0)
print(f"\nTest Accuracy: {accuracy:.4f}")
y_pred = model.predict(x_test)
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

c:\Users\Hashir\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7795 - loss: 0.6820
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9782 - loss: 0.2800
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9923 - loss: 0.1380
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9910 - loss: 0.0901
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9923 - loss: 0.0636
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9897 - loss: 0.0596
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9949 - loss: 0.0404
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9936 - loss: 0.0389
Epoch 9/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9923 - loss: 0.0339
Epoch 10/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9910 - loss: 0.0326
Epoch 11/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9910 - loss: 0.0269
Epoch 12/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9936 - lo